# History, Mem0, and RLM (Chapter 8)

This notebook accompanies Chapter 8 §8.4 of *Context Engineering with DSPy*. It walks through three escalating memory strategies: `dspy.History` for short conversations, Mem0 for persistent cross-session memory, and `dspy.RLM` for programmatic context management over very long inputs.

**Required environment variable**

- `OPENAI_API_KEY` — used by the LM, Mem0's LLM and embedder backends, and the RLM sub-LM.

**Optional service dependency: Mem0**

Mem0 stores memories in a local vector store by default (Qdrant in memory or on disk). For graph memory you can plug in Neo4j via `graph_store` in the config — not required for this notebook. The Mem0 section short-circuits with a clear error if the package or backend is not ready.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
%pip install mem0ai -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## 8.4.1 Conversation history with `dspy.History`

DSPy isolates every call by default. To carry context across turns you declare a `dspy.History` input field on your signature and append each turn's inputs and outputs to a running `History` object.

In [ ]:
class ChatBot(dspy.Signature):
    """You are a helpful assistant. Use the conversation history to maintain context."""
    question: str = dspy.InputField()
    history: dspy.History = dspy.InputField()
    answer: str = dspy.OutputField()


chatbot = dspy.Predict(ChatBot)

# First turn: no history
response1 = chatbot(
    question="What is the capital of France?",
    history=dspy.History(messages=[]),
)
print(response1.answer)  # "Paris"

# Second turn: include previous exchange
history = dspy.History(messages=[
    {"question": "What is the capital of France?", "answer": response1.answer}
])

response2 = chatbot(
    question="What's the population of that city?",
    history=history,
)
print(response2.answer)  # Knows "that city" means Paris

## 8.4.2 Long-term memory with Mem0

Mem0 stores summarised memories outside the context window and retrieves only what is relevant. We wrap its operations as DSPy tools so a ReAct agent can decide when to store and when to recall.

In [ ]:
try:
    from mem0 import Memory

    # Initialize Mem0
    config = {
        "llm": {
            "provider": "openai",
            "config": {"model": "gpt-4o-mini", "temperature": 0.1},
        },
        "embedder": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
    }
    memory = Memory.from_config(config)
    mem0_ready = True
except Exception as exc:
    memory = None
    mem0_ready = False
    print(
        "Could not initialize Mem0. Install with `pip install mem0ai` and ensure "
        "OPENAI_API_KEY is set.\n"
        f"Underlying error: {exc!r}"
    )

In [ ]:
# Create memory tools
def store_memory(content: str, user_id: str = "default") -> str:
    """Store important information in long-term memory for later recall."""
    memory.add(content, user_id=user_id)
    return f"Stored: {content}"


def search_memories(query: str, user_id: str = "default") -> str:
    """Search long-term memory for information relevant to the query."""
    results = memory.search(query, filters={"user_id": user_id}, top_k=5)
    if not results:
        return "No relevant memories found."
    return "\n".join(r["memory"] for r in results["results"])


def get_all_memories(user_id: str = "default") -> str:
    """Retrieve all stored memories for a user."""
    results = memory.get_all(filters={"user_id": user_id})
    if not results:
        return "No memories stored."
    return "\n".join(r["memory"] for r in results["results"])

In [ ]:
class MemoryAgent(dspy.Signature):
    """You are a helpful assistant with long-term memory.
    Store important user information and preferences.
    Search your memory before answering questions about the user."""
    user_input: str = dspy.InputField()
    response: str = dspy.OutputField()


if mem0_ready:
    agent = dspy.ReAct(
        MemoryAgent,
        tools=[store_memory, search_memories, get_all_memories],
        max_iters=6,
    )

    # First conversation
    agent(user_input="My name is Mike and I prefer Python over JavaScript.")
    # Agent stores this preference

    # Later conversation (even days later)
    result = agent(
        user_input="What programming language should you write examples in for me?"
    )
    # Agent searches memory, finds the preference, responds with Python
    print(result.response)
else:
    print("Skipping Mem0 agent — see initialization error above.")

## 8.4.3 Programmatic context management with `dspy.RLM`

Recursive Language Models give the orchestrating model a Python REPL plus a `llm_query()` helper that delegates focused sub-questions to a cheaper LM. This keeps the top-level context small even when the underlying documents are huge.

`dspy.RLM` is still marked experimental, so expect the API to evolve.

In [ ]:
very_long_text = (
    "Q1 customer retention report... "
    "Across enterprise accounts churn dropped 12% quarter over quarter, "
    "driven primarily by the new onboarding revamp. Mid-market churn was flat. "
    "Self-serve churn increased 4%, concentrated in trials that never connected "
    "a data source."
)

rlm = dspy.RLM(
    "documents, question -> answer",
    max_iterations=10,
    max_llm_calls=50,
    sub_lm=dspy.LM("openai/gpt-4o-mini"),
)

result = rlm(
    documents=very_long_text,  # Could be 100K+ characters in production
    question="What were the key findings about customer retention?",
)
print(result.answer)